# 面试问题：怎样从零实现 Greedy、Beam Search、Top-k 与 Top-p 解码？

**一句话回答**：解码器每步读取模型 logits，先按 temperature 和约束变换，再在“确定性搜索”或“截断分布采样”中选 token。Beam 维护累计 log probability 与完成状态并做长度归一化；Top-k 固定候选数，Top-p 保留累计概率最小前缀。线上还需 EOS、最大长度、重复约束、随机种子和停止原因。

本 Notebook 只用 NumPy 实现完整策略，避免把核心行为藏进 `generate()`。

In [ ]:
import hashlib, json, math
import numpy as np

vocab99=["<bos>","我","喜欢","机器","学习","。","<eos>"]; BOS99,EOS99=0,6; V99=len(vocab99)
logits99=np.full((V99,V99),-6.,dtype=float)
for src,pairs in {0:{1:6,3:1},1:{2:6,4:2},2:{3:5.5,4:5},3:{4:6,5:1},4:{5:6,6:2},5:{6:7},6:{6:7}}.items():
    for dst,val in pairs.items(): logits99[src,dst]=val
assert logits99.shape==(7,7)
assert np.argmax(logits99[BOS99])==1 and np.argmax(logits99[5])==EOS99
assert len(set(vocab99))==V99

## 1. 所有策略共享稳定的 log-probability 层

logits 不是概率。先减最大值再做 `logsumexp`，避免大数溢出；序列分数是逐 token log probability 之和，而不是概率相乘。已经生成 EOS 的序列必须冻结，不能继续重复 EOS 来改变长度分数。

In [ ]:
def log_softmax99(z):
    z=np.asarray(z,dtype=float); m=np.max(z,axis=-1,keepdims=True); return z-m-np.log(np.exp(z-m).sum(axis=-1,keepdims=True))
lp99=log_softmax99(np.array([[10000.,9999.,-10000.]])); p99=np.exp(lp99)
assert np.isfinite(lp99).all()
assert np.allclose(p99.sum(-1),1.)
assert np.argmax(lp99)==0 and math.isclose(float(lp99[0,0]),-math.log1p(math.exp(-1)),rel_tol=1e-10)

## 2. Temperature 与 Greedy 的边界

使用 `logits/T`：`T<1` 更尖锐，`T>1` 更平坦；它不改变 argmax，所以纯 Greedy 下 temperature 没有作用。`T→0` 不应直接除零，而应显式走 Greedy 分支。Greedy 快且稳定，但会做局部最优决策。

In [ ]:
def probs99(z,temp=1.):
    if temp<=0: raise ValueError("temperature_contract")
    return np.exp(log_softmax99(np.asarray(z)/temp))
base99=np.array([3.,1.,0.]); cold99=probs99(base99,.5); hot99=probs99(base99,2.)
def entropy99(p): return float(-(p*np.log(p+1e-30)).sum())
assert entropy99(cold99)<entropy99(hot99)
assert np.argmax(cold99)==np.argmax(hot99)==0
try: probs99(base99,0); raise AssertionError("bad temperature accepted")
except ValueError as e: assert str(e)=="temperature_contract"

## 3. Greedy：EOS 与停止原因属于返回合同

每步只保留 argmax，时间复杂度近似 `O(L·V)`。接口不仅返回 token，还应返回累计 logprob、是否命中 EOS 和 `eos/max_length` 停止原因，便于线上诊断截断率。

In [ ]:
def greedy99(start=BOS99,max_new=12):
    seq=[start]; score=0.
    for _ in range(max_new):
        lp=log_softmax99(logits99[seq[-1]]); nxt=int(np.argmax(lp)); seq.append(nxt); score+=float(lp[nxt])
        if nxt==EOS99: return seq,score,"eos"
    return seq,score,"max_length"
greedy_seq99,greedy_score99,greedy_stop99=greedy99()
assert [vocab99[i] for i in greedy_seq99]==["<bos>","我","喜欢","机器","学习","。","<eos>"]
assert greedy_stop99=="eos" and greedy_seq99[-1]==EOS99
assert greedy_score99<=0 and len(greedy_seq99)<=13

## 4. Beam Search：同时维护活跃与已完成假设

每个活跃 beam 展开全部词表，按累计 logprob 保留前 `B` 个。完成序列不再扩展；否则短句会因多乘几个小于 1 的概率而天然占优。真实大模型会先取局部 top candidates，避免构造 `B×V` 的巨大 Python 对象。

In [ ]:
def beam_search99(width=3,max_new=12):
    beams=[([BOS99],0.,False)]
    for _ in range(max_new):
        cand=[]
        for seq,score,done in beams:
            if done: cand.append((seq,score,done)); continue
            lp=log_softmax99(logits99[seq[-1]])
            for tok in range(V99): cand.append((seq+[tok],score+float(lp[tok]),tok==EOS99))
        beams=sorted(cand,key=lambda z:z[1],reverse=True)[:width]
        if all(b[2] for b in beams): break
    return beams
beams99=beam_search99(3); best99=beams99[0]
assert len(beams99)==3 and all(len(b)==3 for b in beams99)
assert best99[0][-1]==EOS99 and best99[2]
assert all(beams99[i][1]>=beams99[i+1][1] for i in range(len(beams99)-1))

## 5. 长度归一化是目标函数的一部分

常用分数为 `logP / length^α` 或 GNMT 的长度惩罚。`α` 越大越偏向长句，但不能把 prompt 与 padding 混进生成长度。面试时应强调：长度惩罚改变排序目标，不是修补 beam 的实现细节，因此必须记录进请求与实验配置。

In [ ]:
def normalized99(logp,length,alpha):
    if length<=0 or alpha<0: raise ValueError("length_contract")
    return logp/(length**alpha)
short_raw99,long_raw99=-1.0,-1.2
assert short_raw99>long_raw99
assert normalized99(short_raw99,2,1)<normalized99(long_raw99,5,1)
try: normalized99(-1,0,.6); raise AssertionError("bad length accepted")
except ValueError as e: assert str(e)=="length_contract"

## 6. Top-k：固定候选规模后重新归一化

先保留 logits 最大的 `k` 个，其余设为负无穷，再 softmax/sample。`k=1` 等价于 Greedy；`k=V` 等价于原分布采样。固定 k 在分布很尖或很平时保留的概率质量差异很大。

In [ ]:
def topk_probs99(z,k,temp=1.):
    z=np.asarray(z,float)
    if not 1<=k<=z.size: raise ValueError("k_contract")
    keep=np.argpartition(z,-k)[-k:]; masked=np.full_like(z,-np.inf); masked[keep]=z[keep]/temp; return np.exp(log_softmax99(masked))
tk199=topk_probs99(base99,1); tk299=topk_probs99(base99,2)
assert np.count_nonzero(tk199)==1 and np.argmax(tk199)==0
assert np.count_nonzero(tk299)==2 and math.isclose(float(tk299.sum()),1.)
try: topk_probs99(base99,4); raise AssertionError("bad k accepted")
except ValueError as e: assert str(e)=="k_contract"

## 7. Top-p / nucleus：保留累计概率最小前缀

对概率降序排列，保留刚好让累计质量达到 `p` 的最短前缀，边界 token 必须包含，再重新归一化。它能随分布形状动态调整候选数。实践常组合 temperature、top-k 上限和 top-p，但顺序必须固定。

In [ ]:
def topp_probs99(z,p,temp=1.):
    if not 0<p<=1: raise ValueError("p_contract")
    prob=probs99(z,temp); order=np.argsort(-prob); cumulative=np.cumsum(prob[order]); count=int(np.searchsorted(cumulative,p,side="left"))+1; keep=order[:count]; out=np.zeros_like(prob); out[keep]=prob[keep]; return out/out.sum(),keep
tp_small99,keep_small99=topp_probs99(base99,.7); tp_all99,keep_all99=topp_probs99(base99,1.)
assert len(keep_small99)==1 and np.argmax(tp_small99)==0
assert len(keep_all99)==3 and np.allclose(tp_all99,probs99(base99))
assert math.isclose(float(tp_small99.sum()),1.)

## 8. 重复约束、随机性与可观测性

repetition penalty 会改 logits；no-repeat n-gram 则根据前缀硬屏蔽会形成重复 n-gram 的 token。约束过强可能把所有 token 屏蔽，必须保留回退。采样器使用请求级 RNG，避免并发请求互相消耗全局随机流；响应记录 seed 与停止原因以便复现。

In [ ]:
def banned_ngram99(seq,n):
    if n<=1: return set(seq)
    prefix=tuple(seq[-(n-1):]); return {seq[i+n-1] for i in range(len(seq)-n+1) if tuple(seq[i:i+n-1])==prefix}
assert banned_ngram99([1,2,1,2],2)=={1}
dist99,_=topp_probs99(logits99[BOS99],.95); draw_a99=np.random.default_rng(99).choice(V99,size=8,p=dist99); draw_b99=np.random.default_rng(99).choice(V99,size=8,p=dist99)
manifest99={"strategy":"top_p","temperature":.8,"top_p":.95,"seed":99,"eos":EOS99,"max_new_tokens":64}; digest99=hashlib.sha256(json.dumps(manifest99,sort_keys=True).encode()).hexdigest()
assert np.array_equal(draw_a99,draw_b99)
assert len(digest99)==64 and manifest99["eos"]==6

## 面试总结

一套强回答应分成 **稳定 logprob → temperature → Greedy → Beam 状态与长度目标 → Top-k → Top-p → 重复约束 → seed/EOS/停止原因**。核心取舍是确定性、质量、多样性、延迟和可复现性，而不是背 `generate` 参数。

延伸阅读：[Hugging Face 生成策略说明](https://huggingface.co/docs/transformers/main/en/generation_strategies)、[Nucleus Sampling 论文](https://arxiv.org/abs/1904.09751)、[GNMT Beam Length Penalty](https://arxiv.org/abs/1609.08144)。